## PTAL (Sumida City) – Data Preparation Manual

#### Created by Tomohiro Ujikawa  
##### (Created in Apr/2025)
- Graduate Student, Luskin School of Public Affairs, University of California, Los Angeles
- Project Researcher, Center for Preventive Medical Sciences, Chiba University 

---

#### Project Description

The term “transit desert” is used to describe areas where there is a significant gap between the demand for and supply of public transportation. To address the transit desert issue, it is crucial to assess access to public transportation and prioritize the development of public transit. In England and India, a tool called the Public Transport Accessibility Level (PTAL) is used to assess accessibility to public transportation. Tokyo is renowned for its highly developed public transportation system. However, mismatches between transportation demand and supply still exist. PTAL has not yet been utilized as a tool for evaluating public transportation in Japan. This project aims to assess public transport accessibility in Sumida City, Tokyo, using PTAL. The PTAL data created in this project is expected to be combined with the other datasets and serve as evidence for improving inequities in public transportation access in Sumida City.

---

#### Data Used

1.	Polygon data of 100m mesh  
    [National Land Numerical Information -> Detailed Land Use Mesh (Area 5339, World Geodetic System, FY2021)](https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-L03-b-2021.html#!)
2.	Location data of bus stops  
    [National Land Numerical Information -> Bus Stop Data -> Tokyo -> FY2022](https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P11-2022.html)
3.	Location data of train stations  
    [National Land Numerical Information -> Railway Data -> FY2023](https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-N02-2023.html)
4.	Boundary data from the 2020 Population Census (Small Area Units – Neighborhoods and Equivalent Divisions)  
    [Boundary Data -> Small Area -> 2020 Population Census -> Small Area (Neighborhoods and Equivalent Divisions) (JGD2000) -> World Geodetic System (latitude/longitude) / Shapefile -> Tokyo -> Sumida City](https://www.e-stat.go.jp/gis/statmap-search?page=1&type=2&aggregateUnitForBoundary=A&toukeiCode=00200521&toukeiYear=2020&serveyId=A002005212020&prefCode=13&coordsys=1&format=shape&datum=2000)
5.	Service frequency information at each SAP (Service Access Points: stations and bus stops), based on online timetable data

---

#### Step 0-1: Creating data linking SAPs (Service Access Points) and mesh centroids (using ArcGIS Pro)

* Import the above datasets (1–4) into ArcGIS.
* Filter the small area polygon data by municipality name to extract only Sumida City, then dissolve the selection to create a unified boundary polygon of Sumida (`Sumida_Border.geojson`).
* Use the mesh data and Sumida City boundary polygon to extract meshes located within Sumida City (select by location → export). Include any meshes that intersect even partially with the boundary (`Sumida_mesh.geojson`).
* Generate point data for the centroids of each mesh (using Feature to Point), and copy the index value into a new column called `MeshID` (`Sumida_mesh_centroids.geojson`).
* Create a buffer polygon that extends 1 km outside the Sumida City boundary. (The furthest SAP is located within 960m from the city boundary, and including unnecessary SAPs will result in wasteful credit consumption during OD Cost Analysis.)
* Use the buffer polygon and the station/bus stop data to extract only the SAP points located within the buffer (select by location). Also copy the index values into `BusStopID` and `StationID` columns respectively.
* Since stations are represented as lines, convert them to point data using Feature to Point.
* Using the centroid points of meshes within Sumida and the SAP points within the buffer, run the **Make OD Cost Matrix Layer** tool to extract all bus stops within 640m and stations within 960m (equivalent to 8 and 12 minutes at 4.8 km/h). Calculate walking distances for each combination and export them as `mesh_to_busstops.geojson` and `mesh_to_stations.geojson`.
* Perform an inner join between `mesh_to_busstops.geojson` and the list of buffered bus stops to extract only the `BusStopID`s used in the OD matrix (apply the same for stations). Export as `Sumida_BusStops_Point.geojson` and `Sumida_Stations_Point.geojson`.

*Note: When creating mesh, station, and bus stop datasets, make sure to include columns for `MeshID`, `StationID`, and `BusStopID`, separate from any index field.*

*Note: If the number of meshes or SAPs is too large to process at once, consider splitting the mesh dataset and running the analysis in parts.*

#### Step 0-2: Create service frequency data for each SAP (manually entered into Excel from online sources)

* Use the point data of bus stops and train stations within the buffer and the data generated from the OD Cost Matrix to perform an inner join (keeping only rows where both datasets match). This is because the data clipped using the buffer polygon does not always represent the actual analysis targets, so a list of stops/stations included in the final OD Cost Matrix analysis should be created.
* Export the resulting list of bus stops and stations from ArcGIS as an Excel file.
* For bus stop data, if multiple bus routes are stored in a single cell, split them so each row corresponds to one bus route. Create a `BusRouteID` column and assign a unique number to each route.
* For station data, although the original data treats the Sobu Line as one, the Rapid and Local services differ in route. Therefore, for example, split Kinshicho Station data into separate rows for Rapid and Local lines. Create a `RailwayID` column and assign route-specific IDs.
* Create a `TransitType` column in both the railway and bus data, labeling train records as 'T' and bus records as 'B'.
* Collect the weekday timetable for each targeted route and enter the service frequency during **8:15 AM to 9:15 AM**. If services operate in both directions, use the direction with the higher frequency. Export these as `BusStops.csv` and `Stations.csv`. The Excel file used for data preparation is located in the `ServiceFrequencyCalc` folder and named `PTAL_TransitFq_Sumida.xlsx`.
  - Choose stations that are centrally located within Sumida Ward and not terminal stations.
  - For lines with express/local classifications, do not apply a single frequency across the whole line. Check whether the station is served by express trains and avoid applying local-only data to express stops.
  - Record in the Excel file which station/stop and direction were used as the reference point for each route's frequency.
  - Some routes have one-way bus stops. Create a column called `NumberOfDirections`, where 0 indicates one-way service and 1 indicates two-way service. (This is not used directly in PTAL calculations but may be useful in the future.)

*Note: Just like for `MeshID`, `StationID`, and `BusStopID`, create separate unique identifiers for bus routes and train routes using `BusRouteID` and `RailwayID` columns, instead of relying solely on index values.*

#### Step 0-3: Loading data

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm

# Load 100m mesh polygon (shapefile) for Sumida City
mesh = gpd.read_file("data/Sumida_mesh.geojson")

# Load shapefile linking mesh centroids to SAPs (Service Access Points: bus stops within 640m and stations within 960m walking distance)
mesh_busstop = gpd.read_file("data/mesh_to_busstops.geojson")
mesh_station = gpd.read_file("data/mesh_to_stations.geojson")

# Load service frequency data (CSV) for each bus route at each bus stop
busstop = pd.read_csv("data/BusStops.csv")
station = pd.read_csv("data/Stations.csv")

---

#### Step 1: Calculate Walking Time to Each SAP

In [ ]:
# The data linking mesh centroids and SAPs includes travel time ('Total_Walk'),
# but it is based on a walking speed of 5 km/h.
# Convert it to travel time using a walking speed of 4.8 km/h based on the distance data ('Total_Kilo').

mesh_busstop['WalkTime'] = mesh_busstop['Total_Kilometers']/4.8*60
mesh_station['WalkTime'] = mesh_station['Total_Kilometers']/4.8*60

#### Step 2: Calculate Scheduled Waiting Time (SWT) at Each SAP

In [ ]:
#SWT(scheduled waiting time) = 0.5 * (60/frequency)
busstop['SWT'] = 0.5*(60/ busstop['PeakHourFq'])
station['SWT'] = 0.5*(60/ station['PeakHourFq'])

#### Step 3: Calculate Average Waiting Time (AWT) at Each SAP

In [ ]:
# AWT (Average Waiting Time) = SWT + reliability buffer
# Add 2 minutes for buses, and 0.75 minutes for trains, subways, and trams as reliability buffer
busstop['AWT'] = busstop['SWT']+2
station['AWT'] = station['SWT']+0.75

#### Step 4: Calculate Total Access Time (TAT) for Each Route at Each SAP

In [ ]:
# Perform join (many-to-many): merge GeoDataFrame on the left with CSV on the right using BusStopID and StationID as keys
joined_mesh_busstop = pd.merge(mesh_busstop, busstop, on='BusStopID', how='left', suffixes=('', '_from_busstop'))
joined_mesh_station = pd.merge(mesh_station, station, on='StationID', how='left', suffixes=('', '_from_station'))

One mesh may be associated with multiple access points (SAPs), and a single transit route may be served by multiple stops/stations.  
In such cases, **only the nearest stop/station should be considered** for calculating TAT (Total Access Time).  
Therefore, in the following steps, TAT will be set to NaN for all but the nearest stop/station.

In [ ]:
# --- For buses ---

# Group by MeshID and BusRouteID, and extract the index of the row with the minimum WalkTime in each group
# (min_indices_bus is a Series that includes only the indices of the closest stops, not others)
min_indices_bus = joined_mesh_busstop.groupby(['MeshID', 'BusRouteID'])['WalkTime'].idxmin()

# Create the 'ClosestInRoute' column and initialize all values to NaN
joined_mesh_busstop['ClosestInRoute'] = np.nan

# Set 'ClosestInRoute' to 1 only for rows indexed by min_indices_bus
joined_mesh_busstop.loc[min_indices_bus, 'ClosestInRoute'] = 1

In [ ]:
# --- For railways ---

# Group by MeshID and RailwayID, and extract the index of the row with the minimum WalkTime in each group
# (min_indices_train is a Series that includes only the indices of the closest stations, not others)
min_indices_train = joined_mesh_station.groupby(['MeshID', 'RailwayID'])['WalkTime'].idxmin()

# Create the 'ClosestInRoute' column and initialize all values to NaN
joined_mesh_station['ClosestInRoute'] = np.nan

# Set 'ClosestInRoute' to 1 only for rows indexed by min_indices_train
joined_mesh_station.loc[min_indices_train, 'ClosestInRoute'] = 1

In [ ]:
# TAT (Total Access Time) = WalkTime + AWT
# Multiplying by 'ClosestInRoute' ensures that TAT is calculated only for the nearest stop/station
# All other entries will result in NaN (due to multiplication by NaN)
joined_mesh_busstop['TAT'] = (joined_mesh_busstop['WalkTime'] + joined_mesh_busstop['AWT']) * joined_mesh_busstop['ClosestInRoute']
joined_mesh_station['TAT'] = (joined_mesh_station['WalkTime'] + joined_mesh_station['AWT']) * joined_mesh_station['ClosestInRoute']

#### Step 5: Calculate Equivalent Doorstep Frequency (EDF) for Each SAP

In [ ]:
# EDF (Equivalent Doorstep Frequency) = 0.5 * (60 / TAT)
# EDF converts Total Access Time (TAT) from time units into frequency units
joined_mesh_busstop['EDF'] = 0.5 * (60 / joined_mesh_busstop['TAT'])
joined_mesh_station['EDF'] = 0.5 * (60 / joined_mesh_station['TAT'])

#### Step 6: Calculate Access Index (AI)

The Access Index (AI) is calculated for each MeshID as follows:  
Among the rows with the same MeshID, the one with the **highest EDF** receives a weight of **1**,  
while **all other EDFs** within that MeshID receive a weight of **0.5**.  
The final AI for each Mesh is the **sum of these weighted EDFs**.

Formula: **AI = max(EDF) + 0.5 * sum(other EDFs within same MeshID)**

##### Calculate AI for buses

In [ ]:
# Add a 'weight' column and initialize all values to 0.5
joined_mesh_busstop['weight'] = 0.5

# Identify the row index with the maximum EDF per MeshID
max_indices_bus = joined_mesh_busstop.groupby('MeshID')['EDF'].idxmax()
max_indices_bus = max_indices_bus.dropna().astype(int)  # Drop NaNs and convert to integer

# Assign a weight of 1.0 only to the row with the maximum EDF
joined_mesh_busstop.loc[max_indices_bus, 'weight'] = 1.0

# Compute the weighted EDF
joined_mesh_busstop['weighted_EDF'] = joined_mesh_busstop['EDF'] * joined_mesh_busstop['weight']

# Calculate AI by summing weighted EDFs per MeshID
AI_bus = joined_mesh_busstop.groupby('MeshID')['weighted_EDF'].sum().reset_index(name='AI')

##### Inspect the resulting AI data

In [ ]:
# Retrieve sets of MeshIDs
ai_set = set(AI_bus['MeshID'])
mesh_set = set(mesh['MeshID'])

# MeshIDs that exist in AI but not in the mesh data
only_in_ai = ai_set - mesh_set

# MeshIDs that exist in mesh data but not in AI
only_in_mesh = mesh_set - ai_set

print("MeshID only in AI:", only_in_ai)
print("MeshID only in mesh:", only_in_mesh)

The 54 MeshIDs found only in the mesh data (and not in AI) are all located along the northeastern riverside area.  
These MeshIDs do not appear in the AI results and should be noted as such.  
List of excluded MeshIDs:  
1089, 1098, 1099, 1108, 1109, 1117, 1118, 1119, 1126, 1127, 1128, 1129, 1139, 1143, 1144, 1147, 1148, 1150, 1151, 1152,  
1287, 1288, 1363, 1364, 1373, 1374, 1381, 1382, 1383, 1390, 1391, 1392, 1399, 1400, 1401, 1407, 1408, 1409, 1415,  
1416, 1417, 1421, 1422, 1423, 1427, 1428, 1429, 1435, 1436, 1439, 1440, 1442, 1443, 1444

In [ ]:
AI_bus['AI'].isnull().sum()

Although the BusStops.csv contains frequency data, there are some rows with missing (null) values.  
There was a concern that certain MeshIDs might only be associated with routes that have no service.  
However, this did **not** occur — all MeshIDs with valid links had routes with service frequency.

##### Calculate AI for Railways

In [ ]:
# Add a 'weight' column and initialize all values to 0.5
joined_mesh_station['weight'] = 0.5

# Identify the row index with the maximum EDF per MeshID
max_indices_train = joined_mesh_station.groupby('MeshID')['EDF'].idxmax()
max_indices_train = max_indices_train.dropna().astype(int)  # Drop NaNs and convert to integer

# Assign a weight of 1.0 only to the row with the maximum EDF
joined_mesh_station.loc[max_indices_train, 'weight'] = 1.0

# Compute the weighted EDF
joined_mesh_station['weighted_EDF'] = joined_mesh_station['EDF'] * joined_mesh_station['weight']

# Calculate AI by summing weighted EDFs per MeshID
AI_train = joined_mesh_station.groupby('MeshID')['weighted_EDF'].sum().reset_index(name='AI')

##### Inspect the resulting AI data

In [ ]:
# Retrieve sets of MeshIDs
ai_set = set(AI_train['MeshID'])
mesh_set = set(mesh['MeshID'])

# MeshIDs that exist in AI but not in mesh data
only_in_ai = ai_set - mesh_set

# MeshIDs that exist in mesh data but not in AI
only_in_mesh = mesh_set - ai_set

print("MeshID only in AI:", only_in_ai)
print("MeshID only in mesh:", only_in_mesh)

The 134 MeshIDs that appear in the mesh data but not in AI are located along the riverside in the northwest, north, and east, as well as in the central southern area.  
These MeshIDs do not produce AI values and should be noted.

List of excluded MeshIDs:  
262, 263, 373, 374, 375, 376, 382, 385, 386, 396, 397, 407, 493, 861, 875, 881, 882, 892, 893, 894, 898, 899, 903, 912, 916, 920, 923, 926, 929,  
1036, 1037, 1038, 1039, 1044, 1045, 1046, 1047, 1048, 1049, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069,  
1076, 1077, 1078, 1079, 1087, 1088, 1089, 1097, 1098, 1099, 1108, 1109, 1117, 1118, 1119, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137,  
1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1154, 1155, 1157, 1158, 1160, 1161, 1163, 1164, 1165, 1166,  
1167, 1168, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1250, 1288, 1343, 1344, 1354, 1363, 1364, 1373, 1374, 1382, 1383, 1392, 1401,  
1430, 1433, 1435, 1436, 1438

#### Merge Mesh Polygon Data with AI Values for Bus and Rail

In [ ]:
# The column name 'AI' exists in both AI_bus and AI_train,
# so rename them to avoid conflicts
AI_bus = AI_bus.rename(columns={'AI': 'AI_bus'})
AI_train = AI_train.rename(columns={'AI': 'AI_train'})

# The 'mesh', AI_bus, and AI_train dataframes all share a 'MeshID' column.
# Use 'MeshID' as the key to merge the AI values into the mesh GeoDataFrame.

# First, perform a left join between mesh and AI_bus
mesh_merged = mesh.merge(AI_bus, on='MeshID', how='left')

# Then, perform another left join with AI_train
mesh_merged = mesh_merged.merge(AI_train, on='MeshID', how='left')

In [ ]:
# Replace null values in 'AI_bus' and 'AI_train' with 0
mesh_merged['AI_bus'] = mesh_merged['AI_bus'].fillna(0)
mesh_merged['AI_train'] = mesh_merged['AI_train'].fillna(0)

# Compute the final Access Index by summing bus and rail values
mesh_merged['AI'] = mesh_merged['AI_bus'] + mesh_merged['AI_train']

#### Map the combined Access Index (AI)

In [ ]:
# Basemap configuration / 背景地図設定 (2026-10-08)
# "carto": preserve Positron; "osm": OpenStreetMap Standard, no API key.
# CARTO keys remain visible in the generated HTML. Restrict the key's Referer hosts.
# CARTO: https://www.carto.com/basemaps/apikey/
# OSM policy: https://operations.osmfoundation.org/policies/tiles/
import os
from getpass import getpass
from pathlib import Path
from urllib.parse import quote

BASEMAP = "carto"  # Change to "osm" to use OpenStreetMap Standard.
MAP_OUTPUT_DIR = Path("maps_fixed/EN")
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OSM_ATTR = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
if BASEMAP == "carto":
    carto_key = os.environ.get("CARTO_BASEMAP_API_KEY", "").strip()
    if not carto_key:
        carto_key = getpass("CARTO Basemaps API key: ").strip()
    if not carto_key:
        raise ValueError("A CARTO Basemaps key is required, or set BASEMAP = 'osm'.")
    BASEMAP_URL = (
        "https://basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}.png?key="
        + quote(carto_key, safe="")
    )
    BASEMAP_ATTR = OSM_ATTR + ' &copy; <a href="https://carto.com/attributions">CARTO</a>'
    BASEMAP_MAX_ZOOM = 20
elif BASEMAP == "osm":
    BASEMAP_URL = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
    BASEMAP_ATTR = OSM_ATTR
    BASEMAP_MAX_ZOOM = 19
else:
    raise ValueError("BASEMAP must be 'carto' or 'osm'.")

# Convert CRS to EPSG:4326 (WGS 84) for web mapping
mesh_merged = mesh_merged.to_crs(epsg=4326)

# Calculate the center coordinates of the map
centroid = mesh_merged.geometry.centroid
center_lat = centroid.y.mean()
center_lon = centroid.x.mean()

# Create a Folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles=BASEMAP_URL, attr=BASEMAP_ATTR, max_zoom=BASEMAP_MAX_ZOOM)

# Get min and max values for AI
min_val = mesh_merged['AI'].min()
max_val = mesh_merged['AI'].max()

# Define a reversed color scale
original_colormap = cm.linear.Spectral_11
reversed_colors = list(original_colormap.colors)[::-1]
colormap = cm.LinearColormap(reversed_colors, vmin=min_val, vmax=max_val)
colormap.caption = 'Access Index (AI)'

# Visualize using GeoJson
folium.GeoJson(
    mesh_merged,
    name="AI Layer",
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['AI']),
        'color': 'None',  # No border line
        'weight': 0,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['MeshID', 'AI'])
).add_to(m)

# Add color scale to map
colormap.add_to(m)

# Save to HTML
m.save(str(MAP_OUTPUT_DIR / "AI.html"))
m

#### Map the Access Index for rail only (AI_train)

In [ ]:
# Recalculate the map center
centroid = mesh_merged.geometry.centroid
center_lat = centroid.y.mean()
center_lon = centroid.x.mean()

# Create a new Folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles=BASEMAP_URL, attr=BASEMAP_ATTR, max_zoom=BASEMAP_MAX_ZOOM)

# Get min and max AI_train values
min_val = mesh_merged['AI_train'].min()
max_val = mesh_merged['AI_train'].max()

# Define reversed color scale
original_colormap = cm.linear.Spectral_11
reversed_colors = list(original_colormap.colors)[::-1]
colormap = cm.LinearColormap(reversed_colors, vmin=min_val, vmax=max_val)
colormap.caption = 'Access Index (Train)'

# Draw using GeoJson
folium.GeoJson(
    mesh_merged,
    name="AI_train Layer",
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['AI_train']),
        'color': 'None',
        'weight': 0,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(fields=['MeshID', 'AI_train'])
).add_to(m)

# Add color scale
colormap.add_to(m)

# Save to HTML
m.save(str(MAP_OUTPUT_DIR / "AI_train.html"))
m

#### Map the Access Index for buses only (AI_bus)

In [ ]:
# Recalculate the map center
centroid = mesh_merged.geometry.centroid
center_lat = centroid.y.mean()
center_lon = centroid.x.mean()

# Create a new Folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles=BASEMAP_URL, attr=BASEMAP_ATTR, max_zoom=BASEMAP_MAX_ZOOM)

# Get min and max AI_bus values
min_val = mesh_merged['AI_bus'].min()
max_val = mesh_merged['AI_bus'].max()

# Define reversed color scale
original_colormap = cm.linear.Spectral_11
reversed_colors = list(original_colormap.colors)[::-1]
colormap = cm.LinearColormap(reversed_colors, vmin=min_val, vmax=max_val)
colormap.caption = 'Access Index (Bus)'

# Draw using GeoJson
folium.GeoJson(
    mesh_merged,
    name="AI_bus Layer",
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['AI_bus']),
        'color': 'None',
        'weight': 0,
        'fillOpacity': 0.7,
    },
    tooltip=folium.GeoJsonTooltip(fields=['MeshID', 'AI_bus'])
).add_to(m)

# Add color scale
colormap.add_to(m)

# Save to HTML
m.save(str(MAP_OUTPUT_DIR / "AI_bus.html"))
m

#### Step 7: Convert AI to PTAL Score

In [ ]:
# Define bin thresholds for classification
bins = [-0.01, 0, 2.5, 5, 10, 15, 20, 25, 40, float('inf')]

# Define PTAL score labels
labels = ['0 (worst)', '1a', '1b', '2', '3', '4', '5', '6a', '6b (best)']

# Create a column to store PTAL scores
mesh_merged['PTAL'] = pd.cut(mesh_merged['AI'], bins=bins, labels=labels)

#### Map the PTAL Score

In [ ]:
# 1. Classify PTAL scores again to ensure consistency
bins = [-0.01, 0, 2.5, 5, 10, 15, 20, 25, 40, float('inf')]
labels = ['0 (worst)', '1a', '1b', '2', '3', '4', '5', '6a', '6b (best)']
mesh_merged['PTAL'] = pd.cut(mesh_merged['AI'], bins=bins, labels=labels)

# 2. Define PTAL color scheme
ptal_colors = {
    '0 (worst)': '#FFFFFF',
    '1a': '#1F3B73',
    '1b': '#2F69BF',
    '2': '#57A0D3',
    '3': '#A5C76F',
    '4': '#FFF24D',
    '5': '#F9C791',
    '6a': '#E23228',
    '6b (best)': '#6B0000'
}

# 3. Create a Folium map
center = mesh_merged.geometry.centroid.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles=BASEMAP_URL, attr=BASEMAP_ATTR, max_zoom=BASEMAP_MAX_ZOOM)

# 4. Add GeoJson layer to map
folium.GeoJson(
    mesh_merged,
    name="PTAL Map",
    style_function=lambda feature: {
        'fillColor': ptal_colors.get(feature['properties']['PTAL'], '#808080'),
        'color': 'None',  # No border line
        'weight': 0,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['MeshID', 'AI', 'PTAL'])
).add_to(m)

# 5. Add legend to map
legend_html = """
<div style="position: fixed; bottom: 20px; left: 20px; width: 160px; 
     background-color: white; z-index:9999; font-size:12px; 
     border:1px solid grey; padding: 8px;">
<b>PTAL Score</b><br>
""" + "".join([
    f'<i style="background:{ptal_colors[label]};width:14px;height:10px;display:inline-block;margin-right:6px;border:1px solid black;"></i>{label}<br>'
    for label in labels
]) + "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# Save the map
m.save(str(MAP_OUTPUT_DIR / "PTAL.html"))
m

#### Save the Final Dataset

In [ ]:
joined_mesh_busstop.to_file("created_data/joined_mesh_busstop.gpkg", driver="GPKG")
joined_mesh_station.to_file("created_data/joined_mesh_station.gpkg", driver="GPKG")
mesh_merged.to_file("created_data/mesh_merged.gpkg", driver="GPKG")

#### Notes and Considerations

- **Train Service Types**
    - Lines such as the Shinjuku Line, Isesaki Line, and Asakusa Line have different service types, such as express and local. Even in the same direction, the final destinations may vary.
    - Since the service frequency differs between express stops and local-only stops, do not apply a uniform frequency to the entire line. Instead, assign separate timetables based on station types.
    - Differences in destination are not considered.
    - Kinshicho Station is the only stop served by both the Sobu Rapid Line and Sobu Local Line. Whether to treat them as separate routes:
        - In the National Land Numerical Information, they are not treated as separate lines and are classified collectively as the "Sobu Line." However, since the actual routes are different, they will be treated as separate lines in this project.

  
- **Direction of Service**
    - The PTAL manual does not consider whether service is one-directional or bidirectional.
    - In this study, however, a column named `NumberOfDirections` will be added for future analyses, assigning 1 for one-way service and 2 for two-way service.
    - Starting bus stops or train stations will be assigned a value of 1 in `NumberOfDirections`.
    - In some cases, such as the Kinshi 11 line, certain segments have one-way stops:
        - After confirming route maps and timetables, assign 1 to those one-way stops in `NumberOfDirections`.
    - For all other stops with bidirectional service, assign a value of 2.
    - Oshiage Station is structurally complex, serving four lines. While it acts as a terminal station for each line (and would technically be classified as one-way service per line), in practice it functions as a bidirectional transfer point due to through-running trains. In this project, it will be treated as four separate one-way lines. Note: This decision could impact the weighting logic in which only the most accessible route receives a weight of 1.  
  

- **Station Grouping**
    - PTAL guidelines mention consolidating nearby bus stops, but do not specify whether railway stations should be grouped.
    - Therefore, railway stations do not need to be merged. However, the National Land Numerical Information provides groupings for stations with the same name located within 300 meters of each other. These groupings will be retained in a column named `StationIDGrouped`.

#### Key Considerations When Adapting the London PTAL Framework to Sumida

- **Walking Speed** (Older Population, Tokyo):  
    - To shorten the maximum walking distance to a service access point (SAP), adjust the walking speed or distance used in the calculations in Step 1 (i.e., modify `Total_Kilometers` in `mesh_to_busstops.geojson` and `mesh_to_stations.geojson`).  
    - To extend the maximum walking distance, you must regenerate the OD Cost Matrix Layer in ArcGIS Pro with updated distance settings.  

- **Delays (Reliability Coefficients)** (Tokyo):  
    - Adjust the coefficients used in Step 3 (average waiting time calculations) to better reflect local conditions.  

- **Time of Day Considered for Service Frequency Data**:  
    - Rush hour periods (Tokyo)  
    - Whether elderly individuals actually travel during rush hour (Older population)  
        - In either case, the data in Step 0-2 (creation of service frequency data for each SAP) must be updated accordingly.  

- **Relative Importance of Bus vs. Rail** (Tokyo):  
    - Consider whether buses and railways should be weighted equally or differently, based on their roles in Tokyo’s transit network.